# Feature Engineering: Venue Features (Percentile-Only)

**Experiment:** Drop raw venue scores (`snip`, `citescore`, `sjr`) and the composite raw score
(`venue_score_composite`) to reduce temporal distribution shift between train (2015-17) and
test (2018-20).

**Motivation (from notebook 53):**
- Raw scores shift heavily: `citescore` KS=0.153, `sjr` KS=0.055, `snip` KS=0.048
- Percentile scores are 3-7x more stable: `citescore_percentile` KS=0.021, etc.
- This causes false positive explosion (562 → 1118 FP/year) as venue scores inflate over time.

**Saves to:** `venue_features_percentile_only.pkl` (does NOT overwrite `venue_features.pkl`)

**Features kept:** `snip_percentile`, `citescore_percentile`, `sjr_percentile`,
`avg_venue_percentile`, `is_top_journal`

**Features dropped:** `snip`, `citescore`, `sjr`, `venue_score_composite`

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

## 1. Load Data

In [ ]:
df = pd.read_pickle('../data/processed/cleaned_data.pkl')
print(f'Dataset: {df.shape}')

## 2. Parse Venue Metrics

In [ ]:
def safe_float(val):
    try:
        return float(val)
    except:
        return np.nan

venue_features = pd.DataFrame(index=df.index)

# Percentile features only — raw scores dropped to reduce temporal distribution shift
venue_features['snip_percentile']      = df['SNIP percentile (publication year) *'].apply(safe_float)
venue_features['citescore_percentile'] = df['CiteScore percentile (publication year) *'].apply(safe_float)
venue_features['sjr_percentile']       = df['SJR percentile (publication year) *'].apply(safe_float)

print('Venue percentile statistics:')
print(venue_features.describe())

## 3. Derived Venue Features

In [ ]:
venue_features['avg_venue_percentile'] = venue_features[[
    'snip_percentile', 'citescore_percentile', 'sjr_percentile'
]].mean(axis=1)

venue_features['is_top_journal'] = (
    (venue_features['snip_percentile']      >= 90) |
    (venue_features['citescore_percentile'] >= 90) |
    (venue_features['sjr_percentile']       >= 90)
).astype(int)

print(f'Top journals: {venue_features["is_top_journal"].sum()}')
print(f'Average venue percentile: {venue_features["avg_venue_percentile"].mean():.2f}')

## 4. Handle Missing Values

In [ ]:
print('Missing values before imputation:')
print(venue_features.isnull().sum())

for col in venue_features.columns:
    if venue_features[col].dtype in ['float64', 'int64']:
        venue_features[col] = venue_features[col].fillna(venue_features[col].median())

print('\nMissing values after imputation:')
print(venue_features.isnull().sum().sum())

## 5. Compare Temporal Stability vs 21b

In [ ]:
from scipy.stats import ks_2samp

# Load old venue features for comparison
old_venue = pd.read_pickle('../data/features/venue_features.pkl')

# Align to train/test years
train_idx = df[df['Year'].isin([2015, 2016, 2017])].index
test_idx  = df[df['Year'].isin([2018, 2019, 2020])].index

print('KS statistic (lower = less temporal shift):')
print(f'{"Feature":<35s}  {"21b (old)":>12s}  {"21c (new)":>12s}')
print('-' * 62)

for col in venue_features.columns:
    new_ks, _ = ks_2samp(venue_features.loc[train_idx, col], venue_features.loc[test_idx, col])
    if col in old_venue.columns:
        old_ks, _ = ks_2samp(old_venue.loc[train_idx, col], old_venue.loc[test_idx, col])
        print(f'{col:<35s}  {old_ks:>12.3f}  {new_ks:>12.3f}')
    else:
        print(f'{col:<35s}  {"N/A":>12s}  {new_ks:>12.3f}')

# Also show dropped raw features for reference
print()
print('Dropped raw features (for reference):')
for col in ['snip', 'citescore', 'sjr', 'venue_score_composite']:
    if col in old_venue.columns:
        ks, _ = ks_2samp(old_venue.loc[train_idx, col], old_venue.loc[test_idx, col])
        print(f'  {col:<33s}  {ks:>12.3f}  {"(dropped)":>12s}')

## 6. Save Features

In [ ]:
output_dir = Path('../data/features')
output_dir.mkdir(parents=True, exist_ok=True)

# Save to a separate file — does NOT overwrite venue_features.pkl
venue_features.to_pickle(output_dir / 'venue_features_percentile_only.pkl')
print(f'Saved to: {output_dir / "venue_features_percentile_only.pkl"}')
print(f'Shape: {venue_features.shape}')
print(f'Features: {list(venue_features.columns)}')

## Summary

In [ ]:
print('=' * 55)
print('VENUE FEATURES (PERCENTILE-ONLY) SUMMARY')
print('=' * 55)
print(f'Total papers: {len(venue_features)}')
print(f'Features: {venue_features.shape[1]} (vs 9 in 21b)')
print(f'  Kept:    snip_percentile, citescore_percentile, sjr_percentile,')
print(f'           avg_venue_percentile, is_top_journal')
print(f'  Dropped: snip, citescore, sjr, venue_score_composite')
print()
print('Next step: run 23_feature_engineering_final.ipynb but load')
print('venue_features_percentile_only.pkl instead of venue_features.pkl')